# IMPORTS:

In [13]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv("cardio_data_processed.csv")

# Preprocessing & Feature Engineering:

In [3]:
df.drop('id', axis=1, inplace = True) # Dropped id becoz its useless for prediction...
df["age"] = (df.age/365).astype(int) # Converting Age in years as its given in days...
df.drop('age_years', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category_encoded', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...

In [4]:
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
df["mean_arterial_pressure"] = df["ap_hi"] + 2*df["ap_lo"]/3

In [5]:
#Checking if there any missing or false value & dropping that...
df[['ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure']].describe()
print((df['pulse_pressure'] < 0).sum()) #Since i got 3 value whose pulse_pressure is < 0 which make no sense as there is an error, so i gonna drop this 3 rows
df = df[df['pulse_pressure'] >= 0] #It removed those 3rows but index still show 0 to 68204 so we need to reset index...
df = df.reset_index(drop=True) #index reset...

3


# Train Test Split:

In [6]:
X = df.drop('cardio', axis = 1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

# RFE Feature Selection:

In [14]:
# Best KNN hyperparams from Optuna (Doesn't show here now as it took exactly 47 mins to execute and i can't afford to execute it again...)
best_params_knn = {
    'n_neighbors': 39,
    'weights': 'uniform',
    'p': 2
}


# Feature count range to test (8-14 features)
feature_counts = range(8, 15)  # 8, 9, 10, 11, 12, 13, 14


# Dictionary to store results
feature_selection_results_knn = {}


# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)


# Loop through each feature count
for n_features in feature_counts:
    print(f"\nTesting with {n_features} features (RFE + KNN)...")
    
    # Store metrics for all folds
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    fold_roc_aucs = []
    
    # CV Loop - FEATURE SELECTION EMBEDDED INSIDE
    fold_idx = 0
    for train_idx, val_idx in cv.split(X_train, y_train):
        fold_idx += 1
        
        # Split data into train and validation
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val  = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val   = y_train.iloc[val_idx]
        
        # Step 1: Scale the training fold
        scaler = StandardScaler()
        X_fold_train_scaled = scaler.fit_transform(X_fold_train)
        X_fold_val_scaled   = scaler.transform(X_fold_val)
        
        # Step 2: Select best features using RFE (LogisticRegression as proxy estimator)
        rfe_estimator = LogisticRegression(max_iter=1000, random_state=10)
        
        rfe = RFE(
            estimator=rfe_estimator,
            n_features_to_select=n_features,
            step=1
        )
        
        X_fold_train_selected = rfe.fit_transform(X_fold_train_scaled, y_fold_train)
        X_fold_val_selected   = rfe.transform(X_fold_val_scaled)
        
        # Step 3: Train KNN model on selected features (using Optuna‑tuned hyperparams)
        model = KNeighborsClassifier(
            n_neighbors=best_params_knn['n_neighbors'],
            weights=best_params_knn['weights'],
            p=best_params_knn['p']
        )
        model.fit(X_fold_train_selected, y_fold_train)
        
        # Step 4: Predict on validation fold
        y_pred = model.predict(X_fold_val_selected)
        
        # If binary classification, use [:,1]; for multi‑class use OVR / appropriate rule
        if len(set(y_fold_val)) == 2:
            y_pred_proba = model.predict_proba(X_fold_val_selected)[:, 1]
        else:
            y_pred_proba = model.predict_proba(X_fold_val_selected).max(axis=1)  # OVR‑style

        # Calculate metrics
        accuracy = accuracy_score(y_fold_val, y_pred)
        precision = precision_score(y_fold_val, y_pred, average='weighted')
        recall = recall_score(y_fold_val, y_pred, average='weighted')
        f1 = f1_score(y_fold_val, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
        
        # Store results
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)
        fold_roc_aucs.append(roc_auc)
    
    # Store mean and std for this feature count
    feature_selection_results_knn[n_features] = {
        'accuracy': np.array(fold_accuracies),
        'precision': np.array(fold_precisions),
        'recall': np.array(fold_recalls),
        'f1': np.array(fold_f1s),
        'roc_auc': np.array(fold_roc_aucs)
    }
    
    # Display results for this feature count
    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy  = np.std(fold_accuracies)
    print(f"  Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")


# ============================================================
# SELECT BEST FEATURE COUNT (based on Accuracy)
# ============================================================

print("\n" + "=" * 80)
print("FEATURE SELECTION RESULTS SUMMARY (K-Nearest Neighbors)")
print("=" * 80)


# Create results dataframe
results_data = {}
for n_features in feature_counts:
    results_data[n_features] = {
        'Accuracy': f"{feature_selection_results_knn[n_features]['accuracy'].mean():.3f} ± {feature_selection_results_knn[n_features]['accuracy'].std():.3f}",
        'Precision': f"{feature_selection_results_knn[n_features]['precision'].mean():.3f} ± {feature_selection_results_knn[n_features]['precision'].std():.3f}",
        'Recall': f"{feature_selection_results_knn[n_features]['recall'].mean():.3f} ± {feature_selection_results_knn[n_features]['recall'].std():.3f}",
        'F1 Score': f"{feature_selection_results_knn[n_features]['f1'].mean():.3f} ± {feature_selection_results_knn[n_features]['f1'].std():.3f}",
        'ROC AUC': f"{feature_selection_results_knn[n_features]['roc_auc'].mean():.3f} ± {feature_selection_results_knn[n_features]['roc_auc'].std():.3f}"
    }

results_df = pd.DataFrame(results_data).T
print("\n")
print(results_df)


# Find best feature count based on Accuracy
best_feature_count_knn = max(
    feature_selection_results_knn.keys(),
    key=lambda x: feature_selection_results_knn[x]['accuracy'].mean()
)

best_accuracy = feature_selection_results_knn[best_feature_count_knn]['accuracy'].mean()
best_accuracy_std = feature_selection_results_knn[best_feature_count_knn]['accuracy'].std()

print("\n" + "=" * 80)
print("BEST FEATURE COUNT (KNN)")
print("=" * 80)
print(f"\nBest Number of Features: {best_feature_count_knn}")
print(f"Best Accuracy: {best_accuracy:.3f} ± {best_accuracy_std:.3f}")
print("\nAll Metrics for Best Feature Count:")
print("-" * 80)
for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    metric_values = feature_selection_results_knn[best_feature_count_knn][metric_name]
    mean = metric_values.mean()
    std = metric_values.std()
    print(f"  {metric_name.upper():10s}: {mean:.3f} ± {std:.3f}")

print("=" * 80)


# Store best feature count for next step
best_features_knn = best_feature_count_knn
print(f"\n✓ Best feature count saved: {best_features_knn}")


Testing with 8 features (RFE + KNN)...
  Accuracy: 0.7261 ± 0.0051

Testing with 9 features (RFE + KNN)...
  Accuracy: 0.7264 ± 0.0045

Testing with 10 features (RFE + KNN)...
  Accuracy: 0.7259 ± 0.0054

Testing with 11 features (RFE + KNN)...
  Accuracy: 0.7256 ± 0.0047

Testing with 12 features (RFE + KNN)...
  Accuracy: 0.7260 ± 0.0045

Testing with 13 features (RFE + KNN)...
  Accuracy: 0.7268 ± 0.0044

Testing with 14 features (RFE + KNN)...
  Accuracy: 0.7272 ± 0.0043

FEATURE SELECTION RESULTS SUMMARY (K-Nearest Neighbors)


         Accuracy      Precision         Recall       F1 Score        ROC AUC
8   0.726 ± 0.005  0.727 ± 0.005  0.726 ± 0.005  0.726 ± 0.005  0.788 ± 0.007
9   0.726 ± 0.005  0.728 ± 0.004  0.726 ± 0.005  0.726 ± 0.005  0.788 ± 0.007
10  0.726 ± 0.005  0.727 ± 0.005  0.726 ± 0.005  0.725 ± 0.005  0.788 ± 0.006
11  0.726 ± 0.005  0.727 ± 0.005  0.726 ± 0.005  0.725 ± 0.005  0.788 ± 0.006
12  0.726 ± 0.005  0.728 ± 0.005  0.726 ± 0.005  0.725 ± 0.005  0.788 

# FINAL KNN PIPELINE:

In [16]:
# Best KNN hyperparams from Optuna
best_params_knn = {
    'n_neighbors': 39,
    'weights': 'uniform',
    'p': 2
}

# Best feature count from RFE
best_features_knn = 14 


# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)


# Store metrics for all folds
fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []
fold_roc_aucs = []


print("\nRunning 5-Fold Stratified Cross-Validation (KNN Final Pipeline)...")
print("-" * 80)
print(f"{'Fold':<6} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'ROC AUC':<12}")
print("-" * 80)


# CV Loop - FEATURE SELECTION (RFE) EMBEDDED INSIDE
fold_idx = 0
for train_idx, val_idx in cv.split(X_train, y_train):
    fold_idx += 1
    
    # Split data into train and validation
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val  = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val   = y_train.iloc[val_idx]
    
    # Step 1: Scale the training fold
    scaler = StandardScaler()
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled   = scaler.transform(X_fold_val)
    
    # Step 2: Select best features using RFE (LogisticRegression as proxy)
    rfe_estimator = LogisticRegression(max_iter=1000, random_state=10)
    
    rfe = RFE(
        estimator=rfe_estimator,
        n_features_to_select=best_features_knn,
        step=1
    )
    
    X_fold_train_selected = rfe.fit_transform(X_fold_train_scaled, y_fold_train)
    X_fold_val_selected   = rfe.transform(X_fold_val_scaled)
    
    # Step 3: Train KNN model on selected features
    model = KNeighborsClassifier(
        n_neighbors=best_params_knn['n_neighbors'],
        weights=best_params_knn['weights'],
        p=best_params_knn['p']
    )
    model.fit(X_fold_train_selected, y_fold_train)
    
    # Step 4: Predict on validation fold
    y_pred = model.predict(X_fold_val_selected)
    
    # If binary classification, use [:,1]; for multiclass use OVR‑style rule
    if len(set(y_fold_val)) == 2:
        y_pred_proba = model.predict_proba(X_fold_val_selected)[:, 1]
    else:
        y_pred_proba = model.predict_proba(X_fold_val_selected).max(axis=1)

    # Calculate metrics
    accuracy = accuracy_score(y_fold_val, y_pred)
    precision = precision_score(y_fold_val, y_pred, average='weighted')
    recall = recall_score(y_fold_val, y_pred, average='weighted')
    f1 = f1_score(y_fold_val, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
    
    # Store results
    fold_accuracies.append(accuracy)
    fold_precisions.append(precision)
    fold_recalls.append(recall)
    fold_f1s.append(f1)
    fold_roc_aucs.append(roc_auc)
    
    # Print fold results
    print(f"{fold_idx:<6} {accuracy:<12.4f} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {roc_auc:<12.4f}")


# Calculate and display summary
print("\n" + "=" * 80)
print("FINAL PIPELINE RESULTS (KNN) - Mean ± Std")
print("=" * 80)

print("\n")
print(f"{'Accuracy':<12s}: {np.mean(fold_accuracies):.3f} ± {np.std(fold_accuracies):.3f}")
print(f"{'Precision':<12s}: {np.mean(fold_precisions):.3f} ± {np.std(fold_precisions):.3f}")
print(f"{'Recall':<12s}: {np.mean(fold_recalls):.3f} ± {np.std(fold_recalls):.3f}")
print(f"{'F1 Score':<12s}: {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"{'ROC AUC':<12s}: {np.mean(fold_roc_aucs):.3f} ± {np.std(fold_roc_aucs):.3f}")


print("\n" + "=" * 80)


# ============================================================
# TRAIN FINAL MODEL ON FULL TRAIN DATA
# ============================================================


print("Training Final KNN Model on full train data...")


# Step 1: Scale the full training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)


# Step 2: Select best features using RFE on full training data (LogisticRegression as proxy)
rfe_estimator = LogisticRegression(max_iter=1000, random_state=10)

final_rfe = RFE(
    estimator=rfe_estimator,
    n_features_to_select=best_features_knn,
    step=1
)

X_train_selected = final_rfe.fit_transform(X_train_scaled, y_train)


# Step 3: Train final KNN model on selected features
final_model_knn = KNeighborsClassifier(
    n_neighbors=best_params_knn['n_neighbors'],
    weights=best_params_knn['weights'],
    p=best_params_knn['p']
)
final_model_knn.fit(X_train_selected, y_train)


print("✓ Final KNN model trained on full training data!")
print("=" * 80)


# Store components for test evaluation
final_scaler_knn      = scaler
final_selector_knn    = final_rfe  # RFE selector (use .transform on X_test_scaled)


Running 5-Fold Stratified Cross-Validation (KNN Final Pipeline)...
--------------------------------------------------------------------------------
Fold   Accuracy     Precision    Recall       F1 Score     ROC AUC     
--------------------------------------------------------------------------------
1      0.7345       0.7361       0.7345       0.7338       0.7986      
2      0.7260       0.7279       0.7260       0.7251       0.7840      
3      0.7290       0.7304       0.7290       0.7283       0.7902      
4      0.7236       0.7252       0.7236       0.7228       0.7845      
5      0.7227       0.7247       0.7227       0.7217       0.7840      

FINAL PIPELINE RESULTS (KNN) - Mean ± Std


Accuracy    : 0.727 ± 0.004
Precision   : 0.729 ± 0.004
Recall      : 0.727 ± 0.004
F1 Score    : 0.726 ± 0.004
ROC AUC     : 0.788 ± 0.006

Training Final KNN Model on full train data...
✓ Final KNN model trained on full training data!


->FINAL PIPELINE FOR RF:
-